In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
from urllib.parse import urlparse, parse_qs

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

/Users/palak/Desktop/Project/Youtube chatHelp/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
def extract_video_id(url):
    parsed_url = urlparse(url)

    if parsed_url.hostname in ["www.youtube.com", "youtube.com"]:
        return parse_qs(parsed_url.query).get("v", [None])[0]

    if parsed_url.hostname == "youtu.be":
        return parsed_url.path.lstrip("/")

    return None

In [38]:
video_url = "https://youtu.be/Yxjh3QBS0_E?si=zxKdoWshG9MVqoBQ"
video_id = extract_video_id(video_url)


from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()

transcript = ytt_api.fetch(video_id)
print(type(transcript))
print(transcript[:100])

<class 'youtube_transcript_api._transcripts.FetchedTranscript'>
[FetchedTranscriptSnippet(text='So you might know that market situation is a bit bad.', start=0.0, duration=2.561), FetchedTranscriptSnippet(text='I mean it is slowly recovering but how did you crack Google in this market situation?', start=2.586, duration=3.654), FetchedTranscriptSnippet(text="Did you think while joining B.Tech that you'll crack Google or Microsoft?", start=6.265, duration=3.412), FetchedTranscriptSnippet(text="When I got Google's offer..... just an hour before I was reading someone's rejection post on LinkedIn", start=9.829, duration=4.085), FetchedTranscriptSnippet(text="that his interview went well but.....so I was a bit hopeless. So first I got HR's call that", start=13.942, duration=3.851), FetchedTranscriptSnippet(text="I'm scheduling a MEET in 10 minutes. So I thought that they're going to conduct further rounds.", start=18.057, duration=3.929), FetchedTranscriptSnippet(text="So I quickly setup my 

In [39]:
text = " ".join(
    snippet.text
    for snippet in transcript
)

print(text[:1000])

So you might know that market situation is a bit bad. I mean it is slowly recovering but how did you crack Google in this market situation? Did you think while joining B.Tech that you'll crack Google or Microsoft? When I got Google's offer..... just an hour before I was reading someone's rejection post on LinkedIn that his interview went well but.....so I was a bit hopeless. So first I got HR's call that I'm scheduling a MEET in 10 minutes. So I thought that they're going to conduct further rounds. So I quickly setup my laptop, took my copy & pen - then they told me. So I couldn't even feel for sometime that I cracked an internship. Programming Language is one thing, people learn it. But there're lot of challenges during DSA. People get stuck in between. Initially, I would just watch lectures & skip it. Then when I slowly felt that things were getting harder when I would watch lectures, I would make my own notes. I would copy the same solution, I would dry run it 2-3 times. So I was a 

In [40]:
from langchain_core.documents import Document

doc = Document(
    page_content=text,
    metadata={
        "source": video_url,
        "video_id": video_id
    }
)

print(doc.page_content[:500])
print(doc.metadata)

So you might know that market situation is a bit bad. I mean it is slowly recovering but how did you crack Google in this market situation? Did you think while joining B.Tech that you'll crack Google or Microsoft? When I got Google's offer..... just an hour before I was reading someone's rejection post on LinkedIn that his interview went well but.....so I was a bit hopeless. So first I got HR's call that I'm scheduling a MEET in 10 minutes. So I thought that they're going to conduct further roun
{'source': 'https://youtu.be/Yxjh3QBS0_E?si=zxKdoWshG9MVqoBQ', 'video_id': 'Yxjh3QBS0_E'}


In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = splitter.split_documents([doc])

print(len(chunks))
print(chunks[0].page_content)
print(chunks[0].metadata)

33
So you might know that market situation is a bit bad. I mean it is slowly recovering but how did you crack Google in this market situation? Did you think while joining B.Tech that you'll crack Google or Microsoft? When I got Google's offer..... just an hour before I was reading someone's rejection post on LinkedIn that his interview went well but.....so I was a bit hopeless. So first I got HR's call that I'm scheduling a MEET in 10 minutes. So I thought that they're going to conduct further rounds. So I quickly setup my laptop, took my copy & pen - then they told me. So I couldn't even feel for sometime that I cracked an internship. Programming Language is one thing, people learn it. But there're lot of challenges during DSA. People get stuck in between. Initially, I would just watch lectures & skip it. Then when I slowly felt that things were getting harder when I would watch lectures, I would make my own notes. I would copy the same solution, I would dry run it 2-3 times. So I was

In [42]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6266.33it/s]


In [44]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="youtube_video_rag"
)

In [45]:
query = "What is this video mainly about?"

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

relevant_docs = retriever.invoke(query)

for i, doc in enumerate(relevant_docs, 1):
    print(f"\n--- Relevant Chunk {i} ---")
    print(doc.page_content[:800])


--- Relevant Chunk 1 ---
are off-campus offers, which requires a good resume. Then we'll also know her strategy that how did she approach the companies So this is a very important podcast for people who're preparing for any company. Watch the podcast till the end. Listen each & everything carefully. And implement it as well. Because you often listen to the podcast, then leave & you don't implement those things. It is very important that you make notes, you implement all these things. Watch the podcast till the end & do subscribe to the channel, if you're new here. Like these videos, if you want more such videos in future. So let's start the podcast. Hi, Shreya! How are you? Hi, Fraz! I'm good, what about you? I'm also doing good! So Shreya, before we start the podcast. Shall we have your small introduction for our

--- Relevant Chunk 2 ---
are off-campus offers, which requires a good resume. Then we'll also know her strategy that how did she approach the companies So this is a very im

In [46]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [67]:
query = "how to get internship at Microsoft and Google"

relevant_docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content for doc in relevant_docs
)

In [68]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Answer the user's question using only the context given below.
If the answer is not present in the context, say:
"I don't know based on the video transcript."

Context:
{context}

Question:
{question}
""")

In [69]:
chain = prompt | llm | StrOutputParser()

In [70]:
answer = chain.invoke({
    "context": context,
    "question": query
})

print(answer)

To get an internship at both Microsoft and Google, here are some general tips:

For Microsoft Internship:

1. **Apply through the official career portal**: You mentioned that you applied for Microsoft's internship through their career portal with the same resume as your Google application.
2. **Check for openings regularly**: Make sure to check the Microsoft career portal frequently for available internships and apply as soon as you see an opening that matches your skills and interests.
3. **Prepare for online assessments**: As you experienced, Microsoft's OA (Online Assessment) consists of coding questions, so it's essential to prepare for these types of assessments.

For Google Internship:

1. **Apply through the official website**: You mentioned that you applied for Google's internship after a few days of applying with your resume.
2. **Tailor your application materials**: Ensure that your resume and cover letter are tailored to the specific internship you're applying for, highlight